# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


In [14]:
# Solution
class Matrix:

    def __init__(self, *args):

        if len(args) == 2:
            n = args[0]
            m = args[1]

            if type(n) != int or type(m) != int:
                raise TypeError("Matrix size must be integers.")

            if n <= 0 or m <= 0:
                raise ValueError("Matrix size must be positive.")

            self.rows = n
            self.cols = m
            self.data = []

            for i in range(n):
                row = []
                for j in range(m):
                    row.append(0)
                self.data.append(row)

        elif len(args) == 1:
            values = args[0]

            if type(values) != list:
                raise TypeError("Input must be a list of lists.")

            if len(values) == 0:
                raise ValueError("Matrix cannot be empty.")

            if type(values[0]) != list:
                raise TypeError("Input must be a list of lists.")

            row_length = len(values[0])

            if row_length == 0:
                raise ValueError("Rows cannot be empty.")

            for i in range(len(values)):
                if type(values[i]) != list:
                    raise TypeError("Input must be a list of lists.")
                if len(values[i]) != row_length:
                    raise ValueError("All rows must have same number of columns.")

            self.rows = len(values)
            self.cols = row_length
            self.data = []

            for i in range(self.rows):
                new_row = []
                for j in range(self.cols):
                    new_row.append(values[i][j])
                self.data.append(new_row)

        else:
            raise TypeError("Use Matrix(n, m) or Matrix(list_of_lists).")

    def __getitem__(self, key):

        if type(key) == int:
            return self.data[key]

        elif type(key) == tuple:
            if type(key[0]) == slice or type(key[1]) == slice:
                row_slice = key[0]
                col_slice = key[1]

                start_row = row_slice.start
                stop_row = row_slice.stop
                start_col = col_slice.start
                stop_col = col_slice.stop

                if start_row is None:
                    start_row = 0
                if stop_row is None:
                    stop_row = self.rows
                if start_col is None:
                    start_col = 0
                if stop_col is None:
                    stop_col = self.cols

                return self.block(start_row, stop_row, start_col, stop_col)

            else:
                i = key[0]
                j = key[1]
                return self.data[i][j]

        else:
            raise TypeError("Invalid index.")

    def __setitem__(self, key, value):
        if type(key) == tuple:
            i = key[0]
            j = key[1]
            self.data[i][j] = value
        else:
            raise TypeError("Use M[i, j] to assign values.")

    def assign(self, other):

        if type(other) == Matrix:

            if self.rows != other.rows or self.cols != other.cols:
                raise ValueError("Matrix sizes must match.")

            for i in range(self.rows):
                for j in range(self.cols):
                    self.data[i][j] = other.data[i][j]

        elif type(other) == list:

            if len(other) != self.rows:
                raise ValueError("Size mismatch.")

            for i in range(len(other)):
                if len(other[i]) != self.cols:
                    raise ValueError("Size mismatch.")

            for i in range(self.rows):
                for j in range(self.cols):
                    self.data[i][j] = other[i][j]

        else:
            raise TypeError("Assignment must be Matrix or list of lists.")

    def __str__(self):

        result = ""
        for i in range(self.rows):
            result = result + str(self.data[i]) + "\n"
        return result

In [15]:
# Test Case

# zero matrix
print("Test 1")
M1 = Matrix(2, 3)
print(M1)

# List of Lists
print("Test 2")
M2 = Matrix([[1, 2, 3], [4, 5, 6]])
print(M2)

# invalid matrix testing
print("Test 2")
M2 = Matrix([[1, 2, 3], [4, 5, 6]])
print(M2)

# test indexing M[i][j]
print("Test 4")
print(M2[1][2])
print("       ")

# test indexing M[i, j]
print("Test 5")
print(M2[1, 2])
print("       ")

# test setting value
print("Test 6")
M2[0, 1] = 99
print(M2)

# matrix to matrix assignment
print("Test 7")
A = Matrix([[1,1,1],[1,1,1]])
B = Matrix([[2,2,2],[2,2,2]])

A.assign(B)
print(A)

# assignment from list
print("Test 8")
A.assign([[9,9,9],[9,9,9]])
print(A)

# test assignment size error
print("Test 9")
try:
    A.assign([[1,2],[3,4]])
except ValueError as e:
    print("Error caught:", e)

Test 1
[0, 0, 0]
[0, 0, 0]

Test 2
[1, 2, 3]
[4, 5, 6]

Test 2
[1, 2, 3]
[4, 5, 6]

Test 4
6
       
Test 5
6
       
Test 6
[1, 99, 3]
[4, 5, 6]

Test 7
[2, 2, 2]
[2, 2, 2]

Test 8
[9, 9, 9]
[9, 9, 9]

Test 9
Error caught: Size mismatch.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * (Extra credit) Modify `__getitem__` implemented above to support slicing.
        

In [29]:
# Solution
class Matrix:

    def __init__(self, *args):

        if len(args) == 2:
            n = args[0]
            m = args[1]

            if type(n) != int or type(m) != int:
                raise TypeError("Matrix size must be integers.")

            if n <= 0 or m <= 0:
                raise ValueError("Matrix size must be positive.")

            self.rows = n
            self.cols = m
            self.data = []

            for i in range(n):
                row = []
                for j in range(m):
                    row.append(0)
                self.data.append(row)

        elif len(args) == 1:
            values = args[0]

            if type(values) != list:
                raise TypeError("Input must be a list of lists.")

            if len(values) == 0:
                raise ValueError("Matrix cannot be empty.")

            if type(values[0]) != list:
                raise TypeError("Input must be a list of lists.")

            row_length = len(values[0])

            if row_length == 0:
                raise ValueError("Rows cannot be empty.")

            for i in range(len(values)):
                if type(values[i]) != list:
                    raise TypeError("Input must be a list of lists.")
                if len(values[i]) != row_length:
                    raise ValueError("All rows must have same number of columns.")

            self.rows = len(values)
            self.cols = row_length
            self.data = []

            for i in range(self.rows):
                new_row = []
                for j in range(self.cols):
                    new_row.append(values[i][j])
                self.data.append(new_row)

        else:
            raise TypeError("Use Matrix(n, m) or Matrix(list_of_lists).")

    def __getitem__(self, key):

        if type(key) == int:
            return self.data[key]

        elif type(key) == tuple:
            if type(key[0]) == slice or type(key[1]) == slice:
                row_slice = key[0]
                col_slice = key[1]

                start_row = row_slice.start
                stop_row = row_slice.stop
                start_col = col_slice.start
                stop_col = col_slice.stop

                if start_row is None:
                    start_row = 0
                if stop_row is None:
                    stop_row = self.rows
                if start_col is None:
                    start_col = 0
                if stop_col is None:
                    stop_col = self.cols

                return self.block(start_row, stop_row, start_col, stop_col)

            else:
                i = key[0]
                j = key[1]
                return self.data[i][j]

        else:
            raise TypeError("Invalid index.")

    def __setitem__(self, key, value):
        if type(key) == tuple:
            i = key[0]
            j = key[1]
            self.data[i][j] = value
        else:
            raise TypeError("Use M[i, j] to assign values.")

    def assign(self, other):

        if type(other) == Matrix:

            if self.rows != other.rows or self.cols != other.cols:
                raise ValueError("Matrix sizes must match.")

            for i in range(self.rows):
                for j in range(self.cols):
                    self.data[i][j] = other.data[i][j]

        elif type(other) == list:

            if len(other) != self.rows:
                raise ValueError("Size mismatch.")

            for i in range(len(other)):
                if len(other[i]) != self.cols:
                    raise ValueError("Size mismatch.")

            for i in range(self.rows):
                for j in range(self.cols):
                    self.data[i][j] = other[i][j]

        else:
            raise TypeError("Assignment must be Matrix or list of lists.")

    def __str__(self):

        result = ""
        for i in range(self.rows):
            result = result + str(self.data[i]) + "\n"
        return result
    def shape(self):
        return (self.rows, self.cols)

    def transpose(self):

        result = Matrix(self.cols, self.rows)

        for i in range(self.rows):
            for j in range(self.cols):
                result.data[j][i] = self.data[i][j]

        return result

    def row(self, n):

        if n < 0 or n >= self.rows:
            raise IndexError("Row index out of range.")

        result = Matrix(1, self.cols)

        for j in range(self.cols):
            result.data[0][j] = self.data[n][j]

        return result

    def column(self, n):

        if n < 0 or n >= self.cols:
            raise IndexError("Column index out of range.")

        result = Matrix(self.rows, 1)

        for i in range(self.rows):
            result.data[i][0] = self.data[i][n]

        return result

    def to_list(self):

        result = []

        for i in range(self.rows):
            new_row = []
            for j in range(self.cols):
                new_row.append(self.data[i][j])
            result.append(new_row)

        return result

    def block(self, n_0, n_1, m_0, m_1):

        if n_0 < 0 or n_1 > self.rows or m_0 < 0 or m_1 > self.cols:
            raise IndexError("Block indices out of range.")

        if n_0 >= n_1 or m_0 >= m_1:
            raise ValueError("Invalid block indices.")

        new_rows = n_1 - n_0
        new_cols = m_1 - m_0

        result = Matrix(new_rows, new_cols)

        for i in range(new_rows):
            for j in range(new_cols):
                result.data[i][j] = self.data[n_0 + i][m_0 + j]

        return result

In [4]:
# Test Case
print("Test 10 - shape")
M = Matrix([[1,2,3],[4,5,6]])
print(M.shape())
print("       ")

print("Test 11 - transpose")
T = M.transpose()
print(T)
print("       ")

print("Test 12 - row")
R = M.row(1)
print(R)
print("       ")

print("Test 13 - column")
C = M.column(2)
print(C)
print("       ")

print("Test 14 - to_list")
L = M.to_list()
print(L)
print("       ")

print("Test 15 - block")
B = Matrix([[1,2,3],
            [4,5,6],
            [7,8,9]])

sub = B.block(0,2,1,3)
print(sub)
print("       ")

print("Test 16 - slicing")
print(B[0:2,1:3])

Test 10 - shape
(2, 3)
       
Test 11 - transpose
[1, 4]
[2, 5]
[3, 6]

       
Test 12 - row
[4, 5, 6]

       
Test 13 - column
[3]
[6]

       
Test 14 - to_list
[[1, 2, 3], [4, 5, 6]]
       
Test 15 - block
[2, 3]
[5, 6]

       
Test 16 - slicing
[2, 3]
[5, 6]



3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [7]:
# Solution
def constant(n, m, c):
    values = []
    for i in range(n):
        row = []
        for j in range(m):
            row.append(float(c))
        values.append(row)
    return Matrix(values)

def zeros(n, m):
    return constant(n, m, 0.0)

def ones(n, m):
    return constant(n, m, 1.0)

In [8]:
# Test Case
C = constant(2, 3, 7)
Z = zeros(3, 4)
O = ones(2, 2)
I_manual = Matrix([[1.0, 0.0, 0.0, 0.0],
                   [0.0, 1.0, 0.0, 0.0],
                   [0.0, 0.0, 1.0, 0.0],
                   [0.0, 0.0, 0.0, 1.0]])

print("Constant matrix 2x3:")
print(C)

print("Zeros matrix 3x4:")
print(Z)

print("Ones matrix 2x2:")
print(O)

print("Manual identity matrix 4x4:")
print(I_manual)

Constant matrix 2x3:
[7.0, 7.0, 7.0]
[7.0, 7.0, 7.0]

Zeros matrix 3x4:
[0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0]

Ones matrix 2x2:
[1.0, 1.0]
[1.0, 1.0]

Manual identity matrix 4x4:
[1.0, 0.0, 0.0, 0.0]
[0.0, 1.0, 0.0, 0.0]
[0.0, 0.0, 1.0, 0.0]
[0.0, 0.0, 0.0, 1.0]



4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

In [9]:
# Solution
class Matrix:

    def __init__(self, *args):

        if len(args) == 2:
            n = args[0]
            m = args[1]

            if type(n) != int or type(m) != int:
                raise TypeError("Matrix size must be integers.")

            if n <= 0 or m <= 0:
                raise ValueError("Matrix size must be positive.")

            self.rows = n
            self.cols = m
            self.data = []

            for i in range(n):
                row = []
                for j in range(m):
                    row.append(0)
                self.data.append(row)

        elif len(args) == 1:
            values = args[0]

            if type(values) != list:
                raise TypeError("Input must be a list of lists.")

            if len(values) == 0:
                raise ValueError("Matrix cannot be empty.")

            if type(values[0]) != list:
                raise TypeError("Input must be a list of lists.")

            row_length = len(values[0])

            if row_length == 0:
                raise ValueError("Rows cannot be empty.")

            for i in range(len(values)):
                if type(values[i]) != list:
                    raise TypeError("Input must be a list of lists.")
                if len(values[i]) != row_length:
                    raise ValueError("All rows must have same number of columns.")

            self.rows = len(values)
            self.cols = row_length
            self.data = []

            for i in range(self.rows):
                new_row = []
                for j in range(self.cols):
                    new_row.append(values[i][j])
                self.data.append(new_row)

        else:
            raise TypeError("Use Matrix(n, m) or Matrix(list_of_lists).")

    def __getitem__(self, key):

        if type(key) == int:
            return self.data[key]

        elif type(key) == tuple:
            if type(key[0]) == slice or type(key[1]) == slice:
                row_slice = key[0]
                col_slice = key[1]

                start_row = row_slice.start
                stop_row = row_slice.stop
                start_col = col_slice.start
                stop_col = col_slice.stop

                if start_row is None:
                    start_row = 0
                if stop_row is None:
                    stop_row = self.rows
                if start_col is None:
                    start_col = 0
                if stop_col is None:
                    stop_col = self.cols

                return self.block(start_row, stop_row, start_col, stop_col)

            else:
                i = key[0]
                j = key[1]
                return self.data[i][j]

        else:
            raise TypeError("Invalid index.")

    def __setitem__(self, key, value):
        if type(key) == tuple:
            i = key[0]
            j = key[1]
            self.data[i][j] = value
        else:
            raise TypeError("Use M[i, j] to assign values.")

    def assign(self, other):

        if type(other) == Matrix:

            if self.rows != other.rows or self.cols != other.cols:
                raise ValueError("Matrix sizes must match.")

            for i in range(self.rows):
                for j in range(self.cols):
                    self.data[i][j] = other.data[i][j]

        elif type(other) == list:

            if len(other) != self.rows:
                raise ValueError("Size mismatch.")

            for i in range(len(other)):
                if len(other[i]) != self.cols:
                    raise ValueError("Size mismatch.")

            for i in range(self.rows):
                for j in range(self.cols):
                    self.data[i][j] = other[i][j]

        else:
            raise TypeError("Assignment must be Matrix or list of lists.")

    def __str__(self):

        result = ""
        for i in range(self.rows):
            result = result + str(self.data[i]) + "\n"
        return result

    def shape(self):
        return (self.rows, self.cols)

    def transpose(self):

        result = Matrix(self.cols, self.rows)

        for i in range(self.rows):
            for j in range(self.cols):
                result.data[j][i] = self.data[i][j]

        return result

    def row(self, n):

        if n < 0 or n >= self.rows:
            raise IndexError("Row index out of range.")

        result = Matrix(1, self.cols)

        for j in range(self.cols):
            result.data[0][j] = self.data[n][j]

        return result

    def column(self, n):

        if n < 0 or n >= self.cols:
            raise IndexError("Column index out of range.")

        result = Matrix(self.rows, 1)

        for i in range(self.rows):
            result.data[i][0] = self.data[i][n]

        return result

    def to_list(self):

        result = []

        for i in range(self.rows):
            new_row = []
            for j in range(self.cols):
                new_row.append(self.data[i][j])
            result.append(new_row)

        return result

    def block(self, n_0, n_1, m_0, m_1):

        if n_0 < 0 or n_1 > self.rows or m_0 < 0 or m_1 > self.cols:
            raise IndexError("Block indices out of range.")

        if n_0 >= n_1 or m_0 >= m_1:
            raise ValueError("Invalid block indices.")

        new_rows = n_1 - n_0
        new_cols = m_1 - m_0

        result = Matrix(new_rows, new_cols)

        for i in range(new_rows):
            for j in range(new_cols):
                result.data[i][j] = self.data[n_0 + i][m_0 + j]

        return result

    def scalarmul(self, c):
        result = Matrix(self.rows, self.cols)
        for i in range(self.rows):
            for j in range(self.cols):
                result.data[i][j] = self.data[i][j] * c
        return result

    def add(self, N):
        if self.rows != N.rows or self.cols != N.cols:
            print('Error: Matrix sizes must match for addition.')
            return None
        result = Matrix(self.rows, self.cols)
        for i in range(self.rows):
            for j in range(self.cols):
                result.data[i][j] = self.data[i][j] + N.data[i][j]
        return result

    def sub(self, N):
        if self.rows != N.rows or self.cols != N.cols:
            print('Error: Matrix sizes must match for subtraction.')
            return None
        result = Matrix(self.rows, self.cols)
        for i in range(self.rows):
            for j in range(self.cols):
                result.data[i][j] = self.data[i][j] - N.data[i][j]
        return result

    def mat_mult(self, N):
        if self.cols != N.rows:
            print('Error: Matrix sizes incompatible for multiplication.')
            return None
        result = Matrix(self.rows, N.cols)
        for i in range(self.rows):
            for j in range(N.cols):
                sum_ = 0
                for k in range(self.cols):
                    sum_ += self.data[i][k] * N.data[k][j]
                result.data[i][j] = sum_
        return result

    def element_mult(self, N):
        if self.rows != N.rows or self.cols != N.cols:
            print('Error: Matrix sizes must match for element-wise multiplication.')
            return None
        result = Matrix(self.rows, self.cols)
        for i in range(self.rows):
            for j in range(self.cols):
                result.data[i][j] = self.data[i][j] * N.data[i][j]
        return result

    def equals(self, N):
        if self.rows != N.rows or self.cols != N.cols:
            return False
        for i in range(self.rows):
            for j in range(self.cols):
                if self.data[i][j] != N.data[i][j]:
                    return False
        return True

    # Operator overloads
    def __add__(self, other):
        return self.add(other)

    def __sub__(self, other):
        return self.sub(other)

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        elif isinstance(other, Matrix):
            return self.mat_mult(other)
        else:
            raise TypeError("Unsupported operand type for *")

    def __rmul__(self, other):
        return self.__mul__(other)

    def __eq__(self, other):
        return self.equals(other)

In [10]:
# Test Case
A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])

print('A + B =')
print(A + B)
print('A - B =')
print(A - B)
print('2 * A =')
print(2 * A)
print('A * 2 =')
print(A * 2)
print('A * B =')
print(A * B)
print('A == B ?')
print(A == B)

A + B =
[6, 8]
[10, 12]

A - B =
[-4, -4]
[-4, -4]

2 * A =
[2, 4]
[6, 8]

A * 2 =
[2, 4]
[6, 8]

A * B =
[19, 22]
[43, 50]

A == B ?
False


5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


In [12]:
# Solution
class Matrix:

    def __init__(self, *args):

        if len(args) == 2:
            n = args[0]
            m = args[1]

            if type(n) != int or type(m) != int:
                raise TypeError("Matrix size must be integers.")

            if n <= 0 or m <= 0:
                raise ValueError("Matrix size must be positive.")

            self.rows = n
            self.cols = m
            self.data = []

            for i in range(n):
                row = []
                for j in range(m):
                    row.append(0)
                self.data.append(row)

        elif len(args) == 1:
            values = args[0]

            if type(values) != list:
                raise TypeError("Input must be a list of lists.")

            if len(values) == 0:
                raise ValueError("Matrix cannot be empty.")

            if type(values[0]) != list:
                raise TypeError("Input must be a list of lists.")

            row_length = len(values[0])

            if row_length == 0:
                raise ValueError("Rows cannot be empty.")

            for i in range(len(values)):
                if type(values[i]) != list:
                    raise TypeError("Input must be a list of lists.")
                if len(values[i]) != row_length:
                    raise ValueError("All rows must have same number of columns.")

            self.rows = len(values)
            self.cols = row_length
            self.data = []

            for i in range(self.rows):
                new_row = []
                for j in range(self.cols):
                    new_row.append(values[i][j])
                self.data.append(new_row)

        else:
            raise TypeError("Use Matrix(n, m) or Matrix(list_of_lists).")

    def __getitem__(self, key):

        if type(key) == int:
            return self.data[key]

        elif type(key) == tuple:
            if type(key[0]) == slice or type(key[1]) == slice:
                row_slice = key[0]
                col_slice = key[1]

                start_row = row_slice.start
                stop_row = row_slice.stop
                start_col = col_slice.start
                stop_col = col_slice.stop

                if start_row is None:
                    start_row = 0
                if stop_row is None:
                    stop_row = self.rows
                if start_col is None:
                    start_col = 0
                if stop_col is None:
                    stop_col = self.cols

                return self.block(start_row, stop_row, start_col, stop_col)

            else:
                i = key[0]
                j = key[1]
                return self.data[i][j]

        else:
            raise TypeError("Invalid index.")

    def __setitem__(self, key, value):
        if type(key) == tuple:
            i = key[0]
            j = key[1]
            self.data[i][j] = value
        else:
            raise TypeError("Use M[i, j] to assign values.")

    def assign(self, other):

        if type(other) == Matrix:

            if self.rows != other.rows or self.cols != other.cols:
                raise ValueError("Matrix sizes must match.")

            for i in range(self.rows):
                for j in range(self.cols):
                    self.data[i][j] = other.data[i][j]

        elif type(other) == list:

            if len(other) != self.rows:
                raise ValueError("Size mismatch.")

            for i in range(len(other)):
                if len(other[i]) != self.cols:
                    raise ValueError("Size mismatch.")

            for i in range(self.rows):
                for j in range(self.cols):
                    self.data[i][j] = other[i][j]

        else:
            raise TypeError("Assignment must be Matrix or list of lists.")

    def __str__(self):

        result = ""
        for i in range(self.rows):
            result = result + str(self.data[i]) + "\n"
        return result
    def shape(self):
        return (self.rows, self.cols)

    def transpose(self):

        result = Matrix(self.cols, self.rows)

        for i in range(self.rows):
            for j in range(self.cols):
                result.data[j][i] = self.data[i][j]

        return result

    def row(self, n):

        if n < 0 or n >= self.rows:
            raise IndexError("Row index out of range.")

        result = Matrix(1, self.cols)

        for j in range(self.cols):
            result.data[0][j] = self.data[n][j]

        return result

    def column(self, n):

        if n < 0 or n >= self.cols:
            raise IndexError("Column index out of range.")

        result = Matrix(self.rows, 1)

        for i in range(self.rows):
            result.data[i][0] = self.data[i][n]

        return result

    def to_list(self):

        result = []

        for i in range(self.rows):
            new_row = []
            for j in range(self.cols):
                new_row.append(self.data[i][j])
            result.append(new_row)

        return result

    def block(self, n_0, n_1, m_0, m_1):

        if n_0 < 0 or n_1 > self.rows or m_0 < 0 or m_1 > self.cols:
            raise IndexError("Block indices out of range.")

        if n_0 >= n_1 or m_0 >= m_1:
            raise ValueError("Invalid block indices.")

        new_rows = n_1 - n_0
        new_cols = m_1 - m_0

        result = Matrix(new_rows, new_cols)

        for i in range(new_rows):
            for j in range(new_cols):
                result.data[i][j] = self.data[n_0 + i][m_0 + j]

        return result
        
    def scalarmul(self, c):
        # Scaling all elements of the matrix by c
        result = Matrix(self.rows, self.cols)
        for i in range(self.rows):
            for j in range(self.cols):
                result.data[i][j] = self.data[i][j] * c
        return result

    def add(self, N):
        if self.rows != N.rows or self.cols != N.cols:
            print('Error: Matrix sizes must match for addition.')
            return None
        result = Matrix(self.rows, self.cols)
        for i in range(self.rows):
            for j in range(self.cols):
                result.data[i][j] = self.data[i][j] + N.data[i][j]
        return result
    
    def sub(self, N):
        if self.rows != N.rows or self.cols != N.cols:
            print('Error: Matrix sizes must match for subtraction.')
            return None
        result = Matrix(self.rows, self.cols)
        for i in range(self.rows):
            for j in range(self.cols):
                result.data[i][j] = self.data[i][j] - N.data[i][j]
        return result
    
    def mat_mult(self, N):
        if self.cols != N.rows:
            print('Error: Matrix sizes incompatible for multiplication.')
            return None
        result = Matrix(self.rows, N.cols)
        for i in range(self.rows):
            for j in range(N.cols):
                sum_ = 0
                for k in range(self.cols):
                    sum_ += self.data[i][k] * N.data[k][j]
                result.data[i][j] = sum_
        return result
    
    def element_mult(self, N):
        # Performing element-wise multiplication
        if self.rows != N.rows or self.cols != N.cols:
            print('Error: Matrix sizes must match for element-wise multiplication.')
            return None
        result = Matrix(self.rows, self.cols)
        for i in range(self.rows):
            for j in range(self.cols):
                result.data[i][j] = self.data[i][j] * N.data[i][j]
        return result
    
    def equals(self, N):
        if self.rows != N.rows or self.cols != N.cols:
            return False
        for i in range(self.rows):
            for j in range(self.cols):
                if self.data[i][j] != N.data[i][j]:
                    return False
        return True 

    def __add__(self, other):
        return self.add(other)

    def __sub__(self, other):
        return self.sub(other)

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        elif isinstance(other, Matrix):
            return self.mat_mult(other)
        else:
            print('Error: unsupported operand for *')
            return None

    def __rmul__(self, other):
        # 2 * M
        return self.__mul__(other)

    def __eq__(self, other):
        # M == N
        return self.equals(other)

    def __str__(self):
        # print nicely
        result = ''
        for row in self.data:
            result += str(row) + '\n'
        return result

In [13]:
# Test Case
A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])

print('A + B =')
print(A + B)
print('A - B =')
print(A - B)
print('2 * A =')
print(2 * A)
print('A * 2 =')
print(A * 2)
print('A * B =')
print(A * B)
print('A == B ?')
print(A == B)

A + B =
[6, 8]
[10, 12]

A - B =
[-4, -4]
[-4, -4]

2 * A =
[2, 4]
[6, 8]

A * 2 =
[2, 4]
[6, 8]

A * B =
[19, 22]
[43, 50]

A == B ?
False


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [15]:
# Solution and Test Case
A = Matrix([[1, 2],
            [3, 4]])

B = Matrix([[0, 1],
            [1, 0]])

C = Matrix([[2, 0],
            [0, 2]])

I = Matrix([[1, 0],
            [0, 1]])

print("Testing Associativity: (AB)C = A(BC)")
left1 = (A * B) * C
right1 = A * (B * C)

print("(AB)C =")
print(left1)
print("A(BC) =")
print(right1)
print("Are they equal?", left1 == right1)
print()


print("Testing Distributive Property: A(B + C) = AB + AC")
left2 = A * (B + C)
right2 = (A * B) + (A * C)

print("A(B + C) =")
print(left2)
print("AB + AC =")
print(right2)
print("Are they equal?", left2 == right2)
print()

print("Testing Non-Commutativity: AB != BA")
AB = A * B
BA = B * A

print("AB =")
print(AB)
print("BA =")
print(BA)
print("Are they equal?", AB == BA)
print()

print("Testing Identity Property: AI = A")
AI = A * I

print("AI =")
print(AI)
print("A =")
print(A)
print("Are they equal?", AI == A)
print()

Testing Associativity: (AB)C = A(BC)
(AB)C =
[4, 2]
[8, 6]

A(BC) =
[4, 2]
[8, 6]

Are they equal? True

Testing Distributive Property: A(B + C) = AB + AC
A(B + C) =
[4, 5]
[10, 11]

AB + AC =
[4, 5]
[10, 11]

Are they equal? True

Testing Non-Commutativity: AB != BA
AB =
[2, 1]
[4, 3]

BA =
[3, 4]
[1, 2]

Are they equal? False

Testing Identity Property: AI = A
AI =
[1, 2]
[3, 4]

A =
[1, 2]
[3, 4]

Are they equal? True

